[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/708san/CV_data/blob/codex/library-ocr-runtime/notebooks/colab/resnet_regression_smoke_test.ipynb)

# ResNet回帰：画像ペアと実測値で試す

このNotebookでは、食前・食後の画像ペアから残量比を予測するResNet回帰を、登録済みのLeFood-Setで試します。データを取得し、trainで学習、validationで確認、testで最終評価します。ColabではT4 GPUを選ぶと実行しやすくなります。

このデータはインドネシアの病院食研究に由来し、日本の学校給食を代表しません。画像予測は計測値の代わりではなく、健康・栄養・医療の判断には使いません。

## 1. ライブラリを入れる

ColabのGPUランタイムを選んでから実行してください。Notebookは固定したGit commitからライブラリを取得します。PyTorch/TorchvisionはColabの環境に合う組み合わせを使います。

In [ ]:
%pip install 'cv-data-lesson[display,torch] @ git+https://github.com/708san/CV_data.git@6a1f49ffa1157edf08a96c5d4e3c4e29b629fb5a'

## 2. 出典とデータを確認する

LeFood-Set v1はCC BY 4.0です。出典・ライセンス・変更内容の表示が必要です。固定Releaseは約53 MBで、画像の取得・展開先はリポジトリ外のキャッシュです。`datasets.get()`を呼ぶと初めて取得します。

In [ ]:
from IPython.display import display
from cvlesson import datasets
from cvlesson.models import ResNetRegressor, available_devices

print(datasets.info("lefood_set"))
data = datasets.get("lefood_set")
frame = data.to_df()
print("分割件数:", data.splits)
print(frame[["food_category", "weight_before_g", "weight_after_g", "consumed_weight_g", "leftover_ratio", "split"]].head())
data.show_examples(n=4, split="train")

## 3. 画像ペアと測定値を用意する

各サンプルは食前・食後の2枚です。ここでは実測値から作られた`leftover_ratio`（食後重量÷食前重量）を予測対象にします。学習を短くするためtrainから固定seedで最大160組を選びます。validationとtestは別splitのまま保ちます。

In [ ]:
train_all = frame[frame["split"] == "train"]
validation = frame[frame["split"] == "validation"].reset_index(drop=True)
test = frame[frame["split"] == "test"].reset_index(drop=True)
train = train_all.sample(n=min(160, len(train_all)), random_state=42).reset_index(drop=True)

X_train = train[["image_before_path", "image_after_path"]]
y_train = train["leftover_ratio"]
X_validation = validation[["image_before_path", "image_after_path"]]
y_validation = validation["leftover_ratio"]
X_test = test[["image_before_path", "image_after_path"]]
y_test = test["leftover_ratio"]
print("使用組数 (train/validation/test):", len(train), len(validation), len(test))
print("利用できる装置:", available_devices())

## 4. ResNet回帰を学習する

ResNet18の特徴抽出部分は固定し、数値を出す出力層を学習します。2枚の画像は別々に特徴へ変換され、組として使われます。

In [ ]:
device = "cuda" if available_devices()["cuda"] else "cpu"
print("学習装置:", device)
model = ResNetRegressor(
    device=device, epochs=3, batch_size=16, image_size=128, random_state=42
)
model.fit(X_train, y_train)
print("学習損失:", [round(value, 4) for value in model.loss_history_])
print("validation R²:", round(model.score(X_validation, y_validation), 3))

## 5. test画像で予測を確認する

R²は実測値と予測値がどの程度対応するかを見る指標です。小さな学習サンプルでは負の値になることもあります。しきい値で成功・失敗を決めず、結果とデータの限界を確認します。回帰出力は0〜1の範囲に制限していないため、範囲外の予測もそのまま表示します。

In [ ]:
test_predictions = model.predict(X_test)
print("test R²:", round(model.score(X_test, y_test), 3))

comparison = test[["food_category", "leftover_ratio", "weight_order_anomaly"]].copy()
comparison["predicted_leftover_ratio"] = test_predictions
comparison["difference"] = comparison["predicted_leftover_ratio"] - comparison["leftover_ratio"]
display(comparison.head(12).round(3))

## 6. 読み取り方

- test R²は学習に使っていない別splitで測っています。値が低い場合も、課題やAPIが動いたかと予測性能を分けて考えます。
- 入力は食前・食後の2枚、正解値は計測由来の残量比です。単一画像の回帰へそのまま置き換えられるとは限りません。
- この研究画像は日本の食事環境を代表せず、画像だけで実際の重量を確定できません。追加データや別環境での確認が必要です。
- 出典：Sari, Yuita Arum; Wani, Yudi Arimba; Nakazawa, Atsushi (2025), *LeFood-Set: Leftovers Food Dataset*, Mendeley Data v1, DOI: 10.17632/cchsk79jkt.1, CC BY 4.0. 教材用に画像の揃う524組を選び、派生列・分割を追加しています。